In [ ]:
import polars as pl
import pandas as pd

import sys
sys.path.append('../../04_utils')

from utils import name_finder, low_context_name_finder

In [ ]:
# Define pathings
data_path = '../../01_data/'
out_path = '../../03_output/01_enriched_results/'

In [3]:
#Load data
pronouns_df = pl.read_csv(data_path + 'das_tfg_pronoun_study.csv', separator= ',' )

pronouns_df.head()

Left,KWIC,Right
str,str,str
"""<s> NARRATOR Yes, indeed. </s>…","""they""","""are locked away, to await the …"
"""<s> NARRATOR Yes, indeed. </s>…","""your""","""fate. </s><s> Only, in the anc…"
"""<s> NARRATOR Yes, indeed. </s>…","""it""","""is stated, that one day an Und…"
"""brands the Undead. </s><s> And…","""it""","""not so that thou art new. </s>…"
"""the Undead. </s><s> And in thi…","""thou""","""art new. </s><s> Thou fared we…"


In [4]:
#Load Character master table
char_master_df = pl.read_csv(data_path + 'das_char_master.csv')\
                   .with_columns(pl.col('Character').str.replace(',','').alias('Character'))

char_master_df.head()

Character,Class,Age
str,str,str
"""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""ANASTACIA OF ASTORA""","""Low""","""Young"""
"""ANDRE OF ASTORA""","""Low""","""Old"""
"""BIG HAТ LOGAN""","""Low""","""Old"""
"""BLACKSMITH VAMOS""","""Low""","""Old"""


In [5]:
#Instantiate list with missing pronouns
missing_pronouns = ['mine', 'mineself',
                    'Mine', 'Mineself']

In [6]:
# Open raw text to look for missing pronouns
with open(data_path + 'Ds corpus input.txt', 'r', encoding='utf-8') as file:
        raw_text = file.read()  # Read the entire content into a single string
        # print(raw_text)

In [7]:
#Construct empty dataframe to input results
left_context = []
pronouns = []
right_context = []
#Search for not found pronouns
for missing in missing_pronouns:
    idx = 0
    context_list = raw_text.split(missing)
    while idx + 1 < len(context_list):
        left_context.append(context_list[idx])
        pronouns.append(missing)
        right_context.append(context_list[idx + 1][:300])
        idx += 1

# Construct dataframe with the found pronouns and contexts, enrich with character name and format contexts to facilitate legibility
missing_pronouns_df = pl.DataFrame().with_columns(pl.Series(left_context).alias('Left'),
                                                 pl.Series(pronouns).alias('KWIC'),
                                                 pl.Series(right_context).alias('Right'))\
                     .with_columns(pl.col('Left').map_elements(name_finder, return_dtype=pl.Utf8).alias('Character'))\
                     .with_columns(pl.col('Left').str.replace_all('<s>','').str.replace_all('</s>',''),
                                   pl.col('Right').str.replace_all('<s>','').str.replace_all('</s>',''))\
                     .with_columns(pl.col('KWIC').str.to_lowercase().alias('KWIC'))\
                     .with_columns(pl.col('Character').forward_fill().alias('Character'))

missing_pronouns_df


Left,KWIC,Right,Character
str,str,str,str
"""NARRATOR Yes, indeed. The Dark…","""mine""",""" senses reveal intruders, then…","""ALVINA OF THE DARKROOT WOOD"""
""" senses reveal intruders, then…","""mine""",""" eyes first set upon thee. Her…","""ALVINA OF THE DARKROOT WOOD"""
""" eyes first set upon thee. Her…","""mine""",""" is yours, but at a price! Not…","""CRESTFALLEN MERCHANT"""
""" is yours, but at a price! Not…","""mine""","""d. They practically queue to e…","""CRESTFALLEN WARRIOR"""
"""d. They practically queue to e…","""mine""","""self, Gwyndolin, and kneel bef…","""DARK SUN GWYNDOLIN"""
…,…,…,…
"""? Thank goodness! I knew he wa…","""mine""",""" will remain in contact. But, …","""SOLAIRE OF ASTORA"""
"""NARRATOR Yes, indeed. The Dark…","""mineself""",""", Gwyndolin, and kneel before …","""DARK SUN GWYNDOLIN"""
""", Gwyndolin, and kneel before …","""mineself""",""", Gwyndolin! Thou shalt not go…","""DARK SUN GWYNDOLIN"""


In [8]:
missing_pronouns_df.filter(pl.col('Character').is_null())

Left,KWIC,Right,Character
str,str,str,str


In [9]:
# Extract the speaking character for each row, turn all KWIC lowercase to avoid redundant values
pronouns_df = pronouns_df.with_columns(pl.col('Left').map_elements(name_finder, return_dtype=pl.Utf8).alias('Character'))\
                         .with_columns(pl.col('KWIC').str.to_lowercase().alias('KWIC'))\
                         .with_columns(pl.col('Character').forward_fill().alias('Character'))\
                         .with_columns(pl.col('Left').str.replace_all('<s>','').str.replace_all('</s>','\n'),
                                       pl.col('Right').str.replace_all('<s>','').str.replace_all('</s>','\n'))\

# Concat with the missing pronouns and trim contexts to facilitate legibility.
pronouns_df = pl.concat([pronouns_df, missing_pronouns_df])\
             .with_columns(pl.col('Left').str.tail(300).alias('Left'),
                           pl.col('Right').str.head(300).alias('Right'))

pronouns_df
                         

Left,KWIC,Right,Character
str,str,str,str
""" NARRATOR Yes, indeed. The D…","""they""","""are locked away, to await the …","""NARRATOR"""
""" NARRATOR Yes, indeed. The D…","""your""","""fate. Only, in the ancient l…","""NARRATOR"""
""" NARRATOR Yes, indeed. The D…","""it""","""is stated, that one day an Und…","""NARRATOR"""
"""ed to the north, where they ar…","""it""","""not so that thou art new. Th…","""ALVINA OF THE DARKROOT WOOD"""
""", where they are locked away, …","""thou""","""art new. Thou fared well to …","""ALVINA OF THE DARKROOT WOOD"""
…,…,…,…
"""ly journey? This pleases me gr…","""mine""",""" will remain in contact. But, …","""SOLAIRE OF ASTORA"""
"""What seeketh thee? Why could t…","""mineself""",""", Gwyndolin, and kneel before …","""DARK SUN GWYNDOLIN"""
""", Gwyndolin, and kneel before …","""mineself""",""", Gwyndolin! Thou shalt not go…","""DARK SUN GWYNDOLIN"""


In [10]:
#Add character class and age information from master table
pronouns_df = pronouns_df.join(char_master_df, on='Character', how = 'left')

pronouns_df

Left,KWIC,Right,Character,Class,Age
str,str,str,str,str,str
""" NARRATOR Yes, indeed. The D…","""they""","""are locked away, to await the …","""NARRATOR""",null,null
""" NARRATOR Yes, indeed. The D…","""your""","""fate. Only, in the ancient l…","""NARRATOR""",null,null
""" NARRATOR Yes, indeed. The D…","""it""","""is stated, that one day an Und…","""NARRATOR""",null,null
"""ed to the north, where they ar…","""it""","""not so that thou art new. Th…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
""", where they are locked away, …","""thou""","""art new. Thou fared well to …","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
…,…,…,…,…,…
"""ly journey? This pleases me gr…","""mine""",""" will remain in contact. But, …","""SOLAIRE OF ASTORA""","""Low""","""Young"""
"""What seeketh thee? Why could t…","""mineself""",""", Gwyndolin, and kneel before …","""DARK SUN GWYNDOLIN""","""High""","""Old"""
""", Gwyndolin, and kneel before …","""mineself""",""", Gwyndolin! Thou shalt not go…","""DARK SUN GWYNDOLIN""","""High""","""Old"""


In [11]:
pronouns_df.filter(pl.col('Character').is_null())

Left,KWIC,Right,Character,Class,Age
str,str,str,str,str,str


In [12]:
# Sanity check for extracted characters
pronouns_df.to_pandas()['Character'].unique()

array(['NARRATOR', 'ALVINA OF THE DARKROOT WOOD', 'ANASTACIA OF ASTORA',
       'ANDRE OF ASTORA', 'BIG HAТ LOGAN', 'BLACKSMITH VAMOS',
       'CRESTFALLEN MERCHANT', 'CRESTFALLEN WARRIOR',
       'CROSSBREED PRISCILLA', 'DARK SUN GWYNDOLIN', 'DARKMOON KNIGHTESS',
       'DARKSTALKER KAATHЕ', 'DOMHNALL OF ZENA', 'DUSK OF OOLACILE',
       'EINGYI OF THE GREAT SWAMP', 'ELIZABETH KEEPER OF THE SANCTUARY',
       'GIANT BLACKSMITH', 'GRIGGS OF VINHEIM',
       'GWYNEVERE PRINCESS OF SUNLIGHT', 'HAWKEYE GOUGН',
       'INGWARD KEEPER OF THE SEAL', 'KINGSEEKER FRAMPТ',
       'LAURENTIUS OF THЕ GREAT SWAMP', 'LAUTREC OF CARIM',
       "LORD''S BLADE CIARAN", 'MARVELOUS CHESTER', 'OSCAR OF ASTORA',
       'OSWALD OF CARIM', 'PETRUS OF THOROLUND', 'QUELANA OF IZALITH',
       'RHEA OF THOROLUND', 'RICKERT OF VINHEIM', 'SHIVA OF THE EAST',
       'SIEGLINDE OF CATARINA', 'SIEGMEYER OF CATARINA', 'HAWK GIRL',
       'SOLAIRE OF ASTORA', 'THE FAIR LADY', 'TRUSTY PATCHES',
       'UNDEAD MERCHANT

In [13]:
# Save enriched dataframe into a csv file.
pronouns_df.write_csv(out_path + 'pronoun_analysis.csv', separator= ';')